# 6 — Change the experiment, not the estimator

*Series: **Bayesian fluorescence decays**, notebook 6 of 8. Builds §2.7 and
§7.2 of `THEORY.md`.*

**What this notebook establishes.** That the bias notebook 5 found is a
property of the *measurement*, not of the fit; that it can be removed by
measuring more, not by estimating better; and how much.

---

## Where the bias came from

Notebook 5 located it: one nearly degenerate direction that trades the sample
scales, the donor-only fraction $x_{D0}$ and the direct-excitation probability
$\varepsilon_{AG}$ against a weak-FRET shoulder at 1.3–1.5 $R/R_0$. Those
parameters do different things to the molecules and nearly the same thing to
the green-pulse histograms:

* a molecule with **no acceptor** contributes an unquenched donor decay;
* a molecule at a **large distance** contributes an almost unquenched donor
  decay;
* an acceptor excited **directly by the green laser** contributes acceptor
  light that did not come from transfer, exactly like a molecule at short
  distance would have contributed a little less of.

No amount of care with the prior fixes an experiment that cannot tell those
apart. A second laser can.

## What the second pulse adds, as an equation

Under interleaved excitation each sample is measured twice per period, and the
two windows share that sample's concentration and acquisition time. So for the
acceptor-carrying sample

$$
\frac{\text{acceptor counts, green pulse, from direct excitation}}{\text{acceptor counts, red pulse}}
\;=\;\varepsilon_{AG},
$$

with the scale cancelling. **The ratio of two windows of the same sample
measures an excitation crosstalk**, which is otherwise a prior. The same
argument applied to the donor-only sample pins its scale against the labelled
one, which is the other half of the degeneracy.

That is the whole idea, and the rest of this notebook measures it.

In [1]:
import sys, time, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
import numpy as np, matplotlib.pyplot as plt, torch
import bd
bd.keep_inline()

S = bd.default_settings(); S['threads'] = 4
model = bd.build_model(S)
L, Ep, rel = model['L'], model['Ep'], model['rel']
p_true = bd.gaussian_truth(model, [0.85, 1.15], [0.55, 0.45], 0.06)
y, lam_true, vals_true = bd.simulate_ensemble(model, S, p_true, x_d0=0.2, photons=3e5, seed=3)
t0 = time.time()
node = bd.fit_at_lambda(model, y, 0.5)
print(f'one node in {time.time()-t0:.0f} s, D/dof {node["dev"]/node["dof"]:.4f}; backend {bd.assert_inline()}')

  maps loaded from /Users/tpeulen/dev/ucfret/investigation/pinn_pR_anisotropy/ckpt/homog/s87_env_128_v2.pt


one node in 25 s, D/dof 1.0669; backend inline


## Asking the question without running the experiment

A design's precision can be computed before any data exist. The expected
information is

$$
\mathcal I(\theta)=\mathbf J^{\!\top}\operatorname{diag}(w/\lambda)\,\mathbf J,
\qquad \mathbf J=\frac{\partial\lambda}{\partial\theta},
$$

where $w_b\in\{0,1\}$ says whether bin $b$ is measured. Add the prior's
curvature, invert, and the diagonal gives the standard deviation each
parameter would have.

Changing **only** $w$ therefore compares experiments. Below, the same
molecules and the same photons, once with both pulses recorded and once with
the red-pulse window discarded — which is what a green-only experiment
measures.

In [2]:
g = node['graph']
th = node['theta']
mask_both = {k: torch.ones(model['n_bin']) for k in g.data_keys}
mask_green = {k: (torch.ones(model['n_bin']) if k[0] == 'D0' else Ep['win_green'].clone())
              for k in g.data_keys}
t0 = time.time()
sd_both, H_both, _ = bd.fisher_sd(model, g, th, mask_both)
sd_green, H_green, _ = bd.fisher_sd(model, g, th, mask_green)
print(f'two designs in {time.time()-t0:.0f} s\n')
show = ['x_d0', 'EX_AG', 'c', 'r0_a', 'g', 'QY_A', 'C_RA', 'log_scale_DA', 'log_scale_A0']
print(f"{'parameter':<15}{'green pulse only':>18}{'both pulses':>14}{'ratio':>9}")
for n in show:
    if n in sd_both:
        print(f'{n:<15}{sd_green[n]:>18.4f}{sd_both[n]:>14.4f}{sd_green[n]/sd_both[n]:>9.2f}')
print('\n(the coefficients c are reported as the mean standard deviation over the spline)')

two designs in 8 s

parameter        green pulse only   both pulses    ratio
x_d0                       0.5727        0.5134     1.12
EX_AG                      0.1469        0.1398     1.05
c                          0.9426        0.9098     1.04
r0_a                       0.1792        0.0749     2.39
g                          0.0044        0.0039     1.12
QY_A                       0.0832        0.0816     1.02
C_RA                       0.0199        0.0198     1.00
log_scale_DA               0.0997        0.0996     1.00
log_scale_A0               0.1232        0.0996     1.24

(the coefficients c are reported as the mean standard deviation over the spline)


### Read that table carefully, because it is easy to over-read

The standard deviations are in the model's own unconstrained coordinates, so
compare the **ratios**, not the values.

What discarding the red-pulse window actually costs, in this design, is the
**acceptor anisotropy** — a factor of 2.4 — and a quarter of the precision on
the red pulse's own scale. Those are the parameters the red window measures
directly. It costs only about 10 % on the donor-only fraction and 5 % on the
direct excitation.

So the second pulse alone, added to an experiment that already has it in the
same histogram, does *not* fix the degeneracy. That is a real result and it
took a moment to accept: the identifiability argument at the top of this
notebook is correct, but the ratio it makes measurable is between the two
windows of the **same sample**, and in this design only the labelled sample is
measured under both lasers. The donor-only reference is not.

## What did fix it

Measure **every reference sample under both lasers**. That turns three
physics scopes into six (notebook 1's table) and gives the window ratio for
the donor-only sample too, which is the half of the degeneracy the calculation
above shows is still open. It was implemented and put through the same
two-hundred-realisation gate as everything else:

| | green pulse only | both pulses |
|---|---|---|
| width bias | 0.0085 | 0.0034 (floor 0.0027) |
| donor-only fraction bias | −0.0042 | −0.0005 |
| width, 95 % coverage | 0.52 | 0.83 |
| far-tail mass, coverage | 0.23 | 0.73 |
| direct excitation $\varepsilon_{AG}$ | prior-dominated | recovered to 0.3 % |

![full PIE](figures/20260908T150202_s88_S15_S15a_onepopulation_1e+06.png)

*One realisation under the six-scope model: the eight histograms with their
weighted residuals, and the recovered distribution.*

The donor-only fraction's bias essentially disappears and two thirds of the
width bias goes with it — because the width bias *was* the donor-only
fraction's, traded through the degenerate direction. The remaining width bias
is close to the floor set by the grid and the smoothing.

**This is the most useful result in the series.** The estimator was already as
good as its information; what was missing was information, and the way to get
it was a second measurement of the reference samples rather than a better
prior. The Fisher calculation above is what makes that checkable in advance —
and, read honestly, it is also what says the cheap version of the idea would
not have been enough.

## The PIE fit as it is run

![PIE](figures/20260906T142417_s88_S9_S9B_merged.png)

*Interleaved excitation fitted the way it is measured: one histogram per
detector holding both pulses, unmasked, with the two scopes summed in the
model. The alternative — splitting the histogram into two windows and calling
them separate channels — throws away the cross-window tails, which are real
photons from real molecules.*

## What comes next

Everything so far has assumed the photons arrive as a histogram. In a
single-molecule experiment they arrive one molecule at a time, and which ones
are used at all is decided by a burst search. Notebook 7 does that, and asks —
by running both — what it costs.

**Established here.** Why the second pulse makes direct excitation and the
donor-only fraction identifiable (THEORY §2.7); the design comparison computed
from the expected information alone, with no data; and the measured effect of
the six-scope experiment on bias and coverage (§7.2).